# Siem Reap Soil Parameters — 04 Evaluation

The test predictions of notebook 03 are scored and the model of each parameter is chosen.

**What you'll learn:**
- how well each parameter can be predicted away from boreholes (Test A), and whether the models beat depth alone and soil + depth
- which model is chosen for each parameter, by a rule fixed before the results
- how skill changes with distance to the nearest borehole and with depth
- how much better a random split (Test B) looks, and why

| Score | Meaning |
|---|---|
| RMSE, MAE | typical error, in the parameter's unit (lower = better) |
| R² | share of the variation explained (1 = perfect, 0 = no better than the test mean) |
| Bias | mean of predicted − observed |
| Spearman | rank correlation |
| Skill vs Depth | 1 − RMSE / RMSE of the Depth reference (> 0: better than depth alone) |
| Skill vs SoilDepth | 1 − RMSE / RMSE of the SoilDepth reference (> 0: location adds something beyond soil type and depth) |

**Model choice per parameter (rule fixed before the results, Test A only):** lowest mean fold RMSE; among the models within one standard error of it, the lowest fold SD, then the lowest MAE. The references compete: if one is chosen, the ML models add nothing for that parameter.

All tables here are aggregates (no borehole codes or locations). **Run time:** under 1 minute.

## 1. Import Libraries and Load the Predictions

In [ ]:
import json, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import utils as U

# Set plot style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100
SOIL_COLORS = {"Clay": "#5B8C3A", "Sand": "#E5B74A", "other": "#9E9E9E", "missing": "#D0D0D0"}

S = U.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| V3 folder:", S["v3_root"])
O = U.out_dir(S, "04_evaluation")
preds = pd.read_csv(U.out_dir(S, "03_test_A_spatial", private=True) / "outer_predictions.csv")
main_block = U.block_sizes(S)[0]
print(f"Test A: {len(preds)} test predictions, {preds.target.nunique()} targets, {preds.model.nunique()} models")

## 2. Scores per Fold, Mean ± SD (Test A)

In [ ]:
folds = U.fold_scores(preds); folds.to_csv(O / "fold_scores.csv", index=False)
summ = U.summary(folds); summ.to_csv(O / "summary.csv", index=False)
cols = ["target", "model", "rmse_mean", "rmse_sd", "mae_mean", "r2_mean", "bias_mean", "spearman_mean",
        "skill_vs_depth", "skill_vs_soildepth"]
summ[summ.block_m == main_block][cols].round(3)

## 3. Results Table and Model Choice

In [ ]:
choice = pd.DataFrame([U.select_model(summ, t, main_block) for t in summ.target.unique()])
choice.to_csv(O / "model_choice.csv", index=False)
s = summ[summ.block_m == main_block].set_index(["target", "model"])
table = pd.DataFrame([{"Parameter": t, "Unit": S["targets"][t]["unit"], "Chosen model": c,
                       "RMSE": s.loc[(t, c), "rmse_mean"], "MAE": s.loc[(t, c), "mae_mean"], "R²": s.loc[(t, c), "r2_mean"],
                       "Skill vs Depth": s.loc[(t, c), "skill_vs_depth"], "Skill vs SoilDepth": s.loc[(t, c), "skill_vs_soildepth"]}
                      for t, c in zip(choice.target, choice.selected)])
table.to_csv(O / "results_table.csv", index=False)
display(choice[["target", "selected", "lowest_mean", "within_1se"]])
table.round(3)

## 4. Skill of Every Model

In [ ]:
sk = summ[summ.block_m == main_block]
sk = sk[~sk.model.isin(U.REFERENCES)]
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, col, title in ((axes[0], "skill_vs_depth", "Skill vs Depth reference"), (axes[1], "skill_vs_soildepth", "Skill vs SoilDepth reference")):
    sns.barplot(data=sk, x="target", y=col, hue="model", ax=ax)
    ax.axhline(0, color="k", lw=0.8); ax.set_title(title); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=45)
axes[1].legend_.remove()
plt.tight_layout(); plt.savefig(O / "skill_by_model.png", dpi=200); plt.show()

**Key observations:** a bar below 0 means the model is worse than the simple reference for that parameter.

## 5. Observed vs Predicted (chosen model, Test A)

In [ ]:
ts = list(choice.target)
ncol = 5; nrow = int(np.ceil(len(ts) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.4 * ncol, 3.4 * nrow))
for ax, (t, mod) in zip(axes.ravel(), zip(choice.target, choice.selected)):
    g = preds[(preds.target == t) & (preds.model == mod) & (preds.block_m == main_block)]
    ax.scatter(g.y, g.y_pred, s=6, alpha=0.35)
    lim = [min(g.y.min(), g.y_pred.min()), max(g.y.max(), g.y_pred.max())]
    ax.plot(lim, lim, "r--", lw=1)
    r2 = U.scores(g.y, g.y_pred)["r2"]
    ax.set_title(f"{t} · {mod}\nR² = {r2:.2f}", fontsize=9)
    ax.set_xlabel(f"observed ({S['targets'][t]['unit']})"); ax.set_ylabel("predicted")
for ax in axes.ravel()[len(ts):]:
    ax.axis("off")
plt.tight_layout(); plt.savefig(O / "observed_vs_predicted.png", dpi=200); plt.show()

## 6. By Distance to the Nearest Training Borehole, and by Depth

In [ ]:
dist = U.distance_table(preds); dist.to_csv(O / "scores_by_distance.csv", index=False)
dep = U.depth_table(preds); dep.to_csv(O / "scores_by_depth.csv", index=False)
chosen = dict(zip(choice.target, choice.selected))
d = dist[(dist.block_m == main_block) & (dist.model == dist.target.map(chosen))]
order = [b for b in ["0-50 m", "50-100 m", "100-200 m", "200-300 m", "300-500 m", "500-1000 m", ">1000 m"] if b in set(d.distance_bin)]
g = sns.catplot(data=d, x="distance_bin", y="r2", col="target", col_wrap=5, kind="point", order=order, height=2.6, aspect=1.2)
g.set_titles("{col_name}"); g.set_xticklabels(rotation=45); g.set_xlabels(""); g.set_ylabels("R²")
plt.savefig(O / "r2_by_distance.png", dpi=200); plt.show()
d.pivot_table(index="target", columns="distance_bin", values="n").reindex(columns=order)

**Key observations:** skill is expected to be highest close to boreholes and to fall with distance (as for the Clay/Sand classification). The number of test samples per band is shown in the table: bands with few samples are not reliable.

## 7. Test A next to Test B
R² and RMSE under the spatial CV (mean of 5 folds) and the random 80:20 split (one split). The gap shows how much of a random-split R² comes from test boreholes having training boreholes nearby.

In [ ]:
f = U.out_dir(S, "03_test_B_random", private=True) / "random_predictions.csv"
if f.exists():
    rnd = pd.read_csv(f)
    b = U.score_table(rnd, U.RUN_KEYS)
    b.to_csv(O / "test_B_scores.csv", index=False)
    a = summ[summ.block_m == main_block][["target", "model", "r2_mean", "rmse_mean"]]
    both = a.merge(b[["target", "model", "r2", "rmse"]], on=["target", "model"])
    both.columns = ["target", "model", "R2 Test A (spatial)", "RMSE Test A", "R2 Test B (random)", "RMSE Test B"]
    both.to_csv(O / "test_A_vs_B.csv", index=False)
    display(both.round(3))
    print("Median distance test borehole -> nearest training borehole: Test A",
          round(preds.drop_duplicates("BH_Code").dist_nearest_train_m.median()), "m | Test B",
          round(rnd.drop_duplicates("BH_Code").dist_nearest_train_m.median()), "m")
else:
    print("Test B not run yet (notebook 03, RUN_TEST_B = True).")

## 8. Summary

In [ ]:
print(f"Finished in {time.time() - T0:.0f} s.")
table.round(3)

**Key findings** (fill in from the outputs above, not assumed):
- parameters the models predict better than soil + depth:
- parameters where a reference was chosen (location adds nothing):
- how skill falls with distance:
- Test A vs Test B gap:

Next (later): **05** settlement and swell from the predicted parameters; **06** 3D map and cross-sections.